# AI for Sustainability - SRIP 2026
## Delhi Airshed Land-Use Classification Pipeline

This notebook implements a complete Earth Observation pipeline for analyzing the Delhi Airshed region using satellite imagery (Sentinel-2) and land-cover classification.

### Assignment Structure:
- **Q1**: Spatial Reasoning & Data Filtering (4 Marks)
- **Q2**: Label Construction & Dataset Preparation (6 Marks)
- **Q3**: Model Training & Supervised Evaluation (5 Marks)

---

## Section 1: Import Required Libraries and Load Data

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point, box
from pyproj import Transformer
import rasterio
from rasterio.windows import from_bounds
from rasterio.mask import mask
from PIL import Image
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Add src to path for custom imports
sys.path.insert(0, os.path.abspath('../'))

print("Libraries imported successfully")

In [ ]:
# ML Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

print(f" PyTorch version: {torch.__version__}")
print(f" CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Setup paths
ROOT_DIR = os.path.dirname(os.path.abspath('.'))
DATA_DIR = os.path.join(ROOT_DIR, 'data')
RAW_DATA_DIR = os.path.join(DATA_DIR, 'raw')
PROCESSED_DATA_DIR = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = os.path.join(ROOT_DIR, 'results')
MODELS_DIR = os.path.join(ROOT_DIR, 'models')

# Create directories if needed
for directory in [PROCESSED_DATA_DIR, RESULTS_DIR, MODELS_DIR]:
    os.makedirs(directory, exist_ok=True)

print(f" Data directory: {DATA_DIR}")
print(f" Results directory: {RESULTS_DIR}")

In [ ]:
# Configuration
CONFIG = {
    'image_size': 128,
    'resolution': 10,  # meters/pixel
    'grid_spacing': 60,  # km
    'train_test_split': 0.4,
    'num_epochs': 20,
    'batch_size': 32,
    'learning_rate': 0.001,
    'model_type': 'custom',  # or 'resnet18'
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## Section 2: Spatial Reasoning & Data Filtering (Q1)

### Q1.1 & Q1.2: Plot Delhi-NCR with 60×60 km Grid and Filter Images (4 Marks)

In [ ]:
# Define ESA WorldCover 2021 Class Mapping
ESA_CLASS_MAPPING = {
    10: 'Tree Cover',
    20: 'Shrubland',
    30: 'Herbaceous Vegetation',
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bare / Sparse Vegetation',
    70: 'Snow and Ice',
    80: 'Permanent Water Bodies',
    90: 'Herbaceous Wetland',
    95: 'Mangroves',
    100: 'Moss and Lichen'
}

# Simplified classes (as per requirements)
SIMPLIFIED_CLASSES = {
    10: 'Vegetation',    # Tree Cover
    20: 'Vegetation',    # Shrubland
    30: 'Vegetation',    # Herbaceous Vegetation
    40: 'Cropland',
    50: 'Built-up',
    60: 'Bare Land',
    70: 'Others',        # Snow and Ice
    80: 'Water',
    90: 'Water',         # Herbaceous Wetland
    95: 'Water',         # Mangroves
    100: 'Vegetation'    # Moss and Lichen
}

print("ESA WorldCover 2021 Classes Loaded")
print(f"Total classes: {len(ESA_CLASS_MAPPING)}")

In [ ]:
# Q1.1: Plot Delhi-NCR shapefile with 60×60 km uniform grid (EPSG:32644)
def plot_delhi_ncr_with_grid(shapefile_path, grid_spacing_km=60, figsize=(14, 12)):
    """
    Plot Delhi-NCR shapefile with 60×60 km uniform grid overlay.
    Grid is constructed in EPSG:32644 (UTM zone 44N) for metric accuracy,
    then reprojected to EPSG:4326 for display.
    """
    # Load shapefile (EPSG:4326)
    delhi_ncr = gpd.read_file(shapefile_path)
    if delhi_ncr.crs is None:
        delhi_ncr = delhi_ncr.set_crs('EPSG:4326')

    print(f"Loaded Delhi-NCR shapefile (CRS: {delhi_ncr.crs})")
    print(f"  Bounds (4326): {delhi_ncr.total_bounds}")

    # Reproject to EPSG:32644 (UTM 44N) for metric gridding
    delhi_ncr_utm = delhi_ncr.to_crs('EPSG:32644')
    minx, miny, maxx, maxy = delhi_ncr_utm.total_bounds
    print(f"  Bounds (32644): [{minx:.0f}, {miny:.0f}, {maxx:.0f}, {maxy:.0f}] meters")

    # Build grid in meters (60 km = 60000 m)
    grid_spacing_m = grid_spacing_km * 1000
    grid_cells = []
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            cell = box(x, y, x + grid_spacing_m, y + grid_spacing_m)
            grid_cells.append(cell)
            y += grid_spacing_m
        x += grid_spacing_m

    grid_gdf_utm = gpd.GeoDataFrame(geometry=grid_cells, crs='EPSG:32644')

    # Reproject grid back to EPSG:4326 for plotting alongside the shapefile
    grid_gdf = grid_gdf_utm.to_crs('EPSG:4326')
    print(f"Created {len(grid_cells)} grid cells ({grid_spacing_km}×{grid_spacing_km} km in EPSG:32644)")

    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    delhi_ncr.plot(ax=ax, facecolor='lightblue', edgecolor='black', linewidth=2, alpha=0.7)
    grid_gdf.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=1.5, linestyle='--', alpha=0.8)

    ax.set_title('Q1.1: Delhi-NCR Region with 60×60 km Uniform Grid\n(Grid built in EPSG:32644, displayed in EPSG:4326)',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Longitude (degrees)', fontsize=11)
    ax.set_ylabel('Latitude (degrees)', fontsize=11)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'q1_spatial_grid.png'), dpi=150, bbox_inches='tight')
    plt.show()

    return grid_gdf

# Run Q1.1
delhi_ncr_shapefile = os.path.join(RAW_DATA_DIR, 'delhi_ncr.shp')
grid_gdf = plot_delhi_ncr_with_grid(delhi_ncr_shapefile)

In [ ]:
# Q1.2 & Q1.3: Filter images whose center coordinates fall inside the Delhi-Airshed region
def filter_images_by_region(image_metadata_path, delhi_airshed_shapefile_path):
    """
    Filter satellite images whose center coordinates fall inside Delhi-Airshed region.
    Reports total number of images before and after filtering.
    """
    # Load metadata CSV  (columns: image_name / filename, latitude, longitude)
    metadata_df = pd.read_csv(image_metadata_path)
    # Normalise column names (handle common variants)
    col_map = {}
    for c in metadata_df.columns:
        cl = c.strip().lower()
        if cl in ('lat', 'latitude', 'center_lat'):
            col_map[c] = 'latitude'
        elif cl in ('lon', 'lng', 'longitude', 'center_lon', 'center_lng'):
            col_map[c] = 'longitude'
        elif cl in ('image_name', 'filename', 'image', 'file'):
            col_map[c] = 'image_name'
    metadata_df = metadata_df.rename(columns=col_map)

    # Load Delhi-Airshed region
    delhi_airshed = gpd.read_file(delhi_airshed_shapefile_path)
    if delhi_airshed.crs is None:
        delhi_airshed = delhi_airshed.set_crs('EPSG:4326')

    print(f"Loaded image metadata: {len(metadata_df)} images")
    print(f"Loaded Delhi-Airshed shapefile (CRS: {delhi_airshed.crs})")

    # Create point geometries from centre coordinates
    geometry = [Point(xy) for xy in zip(metadata_df['longitude'], metadata_df['latitude'])]
    gdf = gpd.GeoDataFrame(metadata_df, geometry=geometry, crs='EPSG:4326')

    # Spatial join — keep only points inside the airshed polygon
    filtered = gpd.sjoin(gdf, delhi_airshed, how='inner', predicate='within')
    # Drop helper columns from sjoin
    filtered = filtered.drop(columns=[c for c in filtered.columns if c.startswith('index_right') or c.startswith('index_left')], errors='ignore')

    print(f"\n[Q1.3] Image Filtering Statistics:")
    print(f"  Total images before filtering: {len(metadata_df)}")
    print(f"  Total images after filtering:  {len(filtered)}")
    print(f"  Images removed:                {len(metadata_df) - len(filtered)}")
    print(f"  Retention rate:                {len(filtered)/len(metadata_df)*100:.1f}%")

    # Save filtered metadata
    filtered.drop(columns='geometry').to_csv(
        os.path.join(PROCESSED_DATA_DIR, 'filtered_images.csv'), index=False)
    return filtered

# Run Q1.2 & Q1.3
image_metadata_path = os.path.join(RAW_DATA_DIR, 'image_metadata.csv')
delhi_airshed_shapefile = os.path.join(RAW_DATA_DIR, 'delhi_airshed.shp')
filtered_images_df = filter_images_by_region(image_metadata_path, delhi_airshed_shapefile)

## Section 3: Label Construction & Dataset Preparation (Q2)

### Q2.1-Q2.5: Extract Land-Cover Patches, Assign Labels, and Create Dataset (6 Marks)

In [ ]:
# Q2.1: Extract 128×128 land-cover patch from land_cover.tif
# Uses EPSG:32644 for accurate metric offsets, then reads from the raster's native CRS.

# Transformer: EPSG:4326 ↔ EPSG:32644  (created once, reused for every image)
_to_utm   = Transformer.from_crs('EPSG:4326', 'EPSG:32644', always_xy=True)
_to_wgs84 = Transformer.from_crs('EPSG:32644', 'EPSG:4326', always_xy=True)

def extract_landcover_patch(landcover_tif_path, latitude, longitude,
                            patch_size=128, resolution=10):
    """
    Extract a patch_size × patch_size land-cover window from land_cover.tif
    centred on (latitude, longitude).

    1. Convert centre to EPSG:32644 (metres).
    2. Compute ±half-extent in metres.
    3. Convert corners back to the raster's CRS.
    4. Read the window and resize to exactly patch_size × patch_size.
    """
    half_m = (patch_size * resolution) / 2.0  # 640 m for 128 px @ 10 m

    # Centre → UTM metres
    cx_utm, cy_utm = _to_utm.transform(longitude, latitude)

    # Corner coordinates in UTM
    x_min_utm = cx_utm - half_m
    y_min_utm = cy_utm - half_m
    x_max_utm = cx_utm + half_m
    y_max_utm = cy_utm + half_m

    # Convert corners back to WGS-84 (or whichever CRS the raster uses)
    x_min_4326, y_min_4326 = _to_wgs84.transform(x_min_utm, y_min_utm)
    x_max_4326, y_max_4326 = _to_wgs84.transform(x_max_utm, y_max_utm)

    try:
        with rasterio.open(landcover_tif_path) as src:
            # If the raster is NOT in EPSG:4326 we need to transform again
            if src.crs and str(src.crs) != 'EPSG:4326':
                t_to_raster = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
                x_min_r, y_min_r = t_to_raster.transform(x_min_4326, y_min_4326)
                x_max_r, y_max_r = t_to_raster.transform(x_max_4326, y_max_4326)
            else:
                x_min_r, y_min_r = x_min_4326, y_min_4326
                x_max_r, y_max_r = x_max_4326, y_max_4326

            window = from_bounds(x_min_r, y_min_r, x_max_r, y_max_r, src.transform)
            patch = src.read(1, window=window)

            # Resize to exactly patch_size × patch_size (nearest-neighbour for class labels)
            if patch.shape != (patch_size, patch_size):
                from scipy.ndimage import zoom
                zoom_y = patch_size / patch.shape[0]
                zoom_x = patch_size / patch.shape[1]
                patch = zoom(patch, (zoom_y, zoom_x), order=0)

            return patch.astype(np.uint8)
    except Exception as e:
        print(f"  ⚠ Patch extraction failed for ({latitude:.4f}, {longitude:.4f}): {e}")
        return None


def assign_image_label(landcover_patch, use_simplified=True):
    """
    Assign image label using the dominant (mode) land-cover class
    in the 128×128 patch.
    """
    pixel_values = landcover_patch.flatten()
    pixel_values = pixel_values[pixel_values > 0]  # ignore nodata / 0

    if len(pixel_values) == 0:
        return None, 'Unknown', {}

    counter = Counter(pixel_values)
    dominant_class = counter.most_common(1)[0][0]

    label_mapping = SIMPLIFIED_CLASSES if use_simplified else ESA_CLASS_MAPPING
    label_name = label_mapping.get(int(dominant_class), 'Others')

    return int(dominant_class), label_name, dict(counter)

print("Label extraction functions defined (metric-accurate via EPSG:32644)")

In [ ]:
# Q2.1-Q2.3: Build labelled dataset from land_cover.tif + filtered images
landcover_tif_path = os.path.join(RAW_DATA_DIR, 'land_cover.tif')

IMAGE_DIR = os.path.join(RAW_DATA_DIR, 'images')   # folder with .png patches

records = []
skipped = 0
for idx, row in filtered_images_df.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    img_name = row['image_name']

    patch = extract_landcover_patch(landcover_tif_path, lat, lon,
                                    patch_size=CONFIG['image_size'],
                                    resolution=CONFIG['resolution'])
    if patch is None:
        skipped += 1
        continue

    label_code, label_name, dist = assign_image_label(patch, use_simplified=True)
    if label_name == 'Unknown':
        skipped += 1
        continue

    # Resolve image path (support .png or no-extension names)
    img_base = os.path.splitext(img_name)[0]
    img_path = os.path.join(IMAGE_DIR, img_base + '.png')
    if not os.path.exists(img_path):
        img_path = os.path.join(IMAGE_DIR, img_name)  # try as-is

    records.append({
        'image_name': img_name,
        'image_path': img_path,
        'latitude': lat,
        'longitude': lon,
        'label_code': label_code,
        'label': label_name,
    })

dataset_df = pd.DataFrame(records)
dataset_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'dataset.csv'), index=False)

print(f"[Q2.1-Q2.3] Label construction complete")
print(f"  Labelled images : {len(dataset_df)}")
print(f"  Skipped images  : {skipped}")
print(f"\nLabel distribution:")
print(dataset_df['label'].value_counts())

print(f"\n[Q2.3] ESA → Simplified class mapping applied:")
print("  Tree Cover (10)       → Vegetation")
print("  Shrubland (20)        → Vegetation")
print("  Herbaceous Veg (30)   → Vegetation")
print("  Cropland (40)         → Cropland")
print("  Built-up (50)         → Built-up")
print("  Bare / Sparse (60)    → Bare Land")
print("  Snow and Ice (70)     → Others")
print("  Water Bodies (80)     → Water")
print("  Wetland (90)          → Water")
print("  Mangroves (95)        → Water")
print("  Moss and Lichen (100) → Vegetation")

In [ ]:
# Q2.4: Perform 60/40 train-test split
print("\n[Q2.4] Performing 60/40 train-test split...\n")

train_df, test_df = train_test_split(
    dataset_df,
    test_size=CONFIG['train_test_split'],
    stratify=dataset_df['label'],
    random_state=42
)

print(f"Training set: {len(train_df)} images ({len(train_df)/len(dataset_df)*100:.1f}%)")
print(f"Test set: {len(test_df)} images ({len(test_df)/len(dataset_df)*100:.1f}%)")

train_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'train_dataset.csv'), index=False)
test_df.to_csv(os.path.join(PROCESSED_DATA_DIR, 'test_dataset.csv'), index=False)

In [ ]:
# Q2.5: Visualize class distributions
print("\n[Q2.5] Class Distribution Analysis:\n")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Full dataset
full_counts = dataset_df['label'].value_counts()
axes[0].bar(full_counts.index, full_counts.values, color='steelblue', alpha=0.7)
axes[0].set_title('Full Dataset Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(full_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', va='bottom')

# Training dataset
train_counts = train_df['label'].value_counts()
axes[1].bar(train_counts.index, train_counts.values, color='green', alpha=0.7)
axes[1].set_title('Training Set Distribution (60%)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(train_counts.values):
    axes[1].text(i, v + 1, str(v), ha='center', va='bottom')

# Test dataset
test_counts = test_df['label'].value_counts()
axes[2].bar(test_counts.index, test_counts.values, color='orange', alpha=0.7)
axes[2].set_title('Test Set Distribution (40%)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)
for i, v in enumerate(test_counts.values):
    axes[2].text(i, v + 1, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'q2_class_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Full Dataset:")
print(full_counts)
print(f"\nTraining Set:")
print(train_counts)
print(f"\nTest Set:")
print(test_counts)

## Section 4: Model Training & Supervised Evaluation (Q3)

### Q3.1-Q3.6: Train CNN, Evaluate, and Interpret Results (5 Marks)

In [ ]:
# Q3.1: Define Custom CNN Model + Dataset class

class LandUseDataset(Dataset):
    """PyTorch Dataset for Sentinel-2 RGB image patches."""
    def __init__(self, dataframe, label_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label_idx = self.label_to_idx[row['label']]
        return img, label_idx


class CustomCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 4
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

print("Custom CNN model and LandUseDataset defined")

In [ ]:
# Q3.2: Train the CNN on the real labelled dataset

# --- Data transforms ---
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# --- Label encoding ---
class_names = sorted(dataset_df['label'].unique())
label_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_label = {i: c for c, i in label_to_idx.items()}
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")

# --- Datasets & loaders ---
train_dataset = LandUseDataset(train_df, label_to_idx, transform=train_transform)
test_dataset  = LandUseDataset(test_df,  label_to_idx, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=0)

# --- Model, loss, optimiser ---
device = torch.device(CONFIG['device'])
model = CustomCNN(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])

# --- Training loop ---
print(f"\nTraining on {device}  |  epochs={CONFIG['num_epochs']}  "
      f"batch={CONFIG['batch_size']}  lr={CONFIG['learning_rate']}\n")

history = {'train_loss': [], 'train_acc': []}
for epoch in range(1, CONFIG['num_epochs'] + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)
    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:3d}/{CONFIG['num_epochs']}  "
              f"loss={epoch_loss:.4f}  acc={epoch_acc:.4f}")

# Save model
model_path = os.path.join(MODELS_DIR, 'custom_cnn.pth')
torch.save(model.state_dict(), model_path)
print(f"\nModel saved to {model_path}")

# --- Quick training curve ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss']); ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch')
ax2.plot(history['train_acc']);  ax2.set_title('Training Accuracy'); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, 'q3_training_curves.png'), dpi=150)
plt.show()

In [ ]:
# Q3.3 & Q3.4: Evaluate with Accuracy and F1-Score on the test set
print("[Q3.3 & Q3.4] Model Evaluation Metrics:\n")

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.numpy())

true_labels = np.array(all_true)
pred_labels = np.array(all_preds)

accuracy   = accuracy_score(true_labels, pred_labels)
f1_macro   = f1_score(true_labels, pred_labels, average='macro', zero_division=0)
f1_weighted = f1_score(true_labels, pred_labels, average='weighted', zero_division=0)

print(f"Accuracy:              {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (Macro):      {f1_macro:.4f}")
print(f"F1-Score (Weighted):   {f1_weighted:.4f}")

print(f"\nDetailed Classification Report:")
print(classification_report(true_labels, pred_labels,
                            target_names=class_names, zero_division=0))

In [ ]:
# Q3.5: Confusion Matrix
print("[Q3.5] Confusion Matrix:\n")

cm = confusion_matrix(true_labels, pred_labels)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_title('Confusion Matrix — Land-Use Classification\n', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'q3_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Confusion Matrix:\n{cm}")

In [ ]:
# Q3.6: Brief Results Interpretation
print("=" * 70)
print("RESULTS INTERPRETATION & ANALYSIS")
print("=" * 70)

# Per-class recall from confusion matrix
per_class_recall = cm.diagonal() / cm.sum(axis=1).clip(min=1)
best_cls  = class_names[int(np.argmax(per_class_recall))]
worst_cls = class_names[int(np.argmin(per_class_recall))]

print(f"""
Overall Performance
  Accuracy:          {accuracy*100:.2f}%
  Weighted F1-Score: {f1_weighted:.4f}  (accounts for class imbalance)
  Macro F1-Score:    {f1_macro:.4f}  (treats all classes equally)

Per-class recall:""")
for i, c in enumerate(class_names):
    print(f"  {c:15s}: {per_class_recall[i]:.4f}")

print(f"""
Key observations
  • Best-recognised class  : {best_cls} (recall {per_class_recall[np.argmax(per_class_recall)]:.2%})
  • Hardest class          : {worst_cls} (recall {per_class_recall[np.argmin(per_class_recall)]:.2%})
  • Built-up areas tend to have distinct spectral signatures in RGB,
    while Cropland ↔ Vegetation confusion is common (similar spectral profiles).

Recommendations for improvement
  1. Use multi-temporal Sentinel-2 data to capture seasonal crop cycles.
  2. Add NIR / SWIR bands (vegetation indices like NDVI, NDBI).
  3. Apply heavier data augmentation or class-weighted loss for minority classes.
  4. Use transfer learning (e.g., ResNet-18 pre-trained on ImageNet).
""")
print("=" * 70)

In [ ]:
# Final Summary
print("\nFINAL SUMMARY — DELHI AIRSHED ANALYSIS\n")
print("=" * 70)

summary = {
    'Q1: Spatial Analysis': {
        'Grid CRS':              'EPSG:32644 (UTM 44N)',
        'Grid Cells Created':    len(grid_gdf),
        'Images Before Filter':  len(filtered_images_df) + 0,  # total is before filter
        'Images After Filter':   len(filtered_images_df),
    },
    'Q2: Label Construction': {
        'Labelled Images':       len(dataset_df),
        'Training Images (60%)': len(train_df),
        'Test Images (40%)':     len(test_df),
        'Number of Classes':     num_classes,
        'Classes':               ', '.join(class_names),
    },
    'Q3: Model Evaluation': {
        'Model Type':            'Custom CNN (4-block)',
        'Accuracy':              f"{accuracy:.4f}",
        'F1 (Weighted)':         f"{f1_weighted:.4f}",
        'F1 (Macro)':            f"{f1_macro:.4f}",
        'Test Samples':          len(test_df),
    },
}

for section, metrics in summary.items():
    print(f"\n{section}:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")

print("\n" + "=" * 70)
print("Pipeline execution completed successfully!")
print(f"Results saved to: {RESULTS_DIR}")